# Agentic RAG

### Agentic RAG

**참고**

- RAG 를 수행하되, Agent 를 활용하여 RAG 를 수행한다면 이를 **Agentic RAG** 라고 부릅니다.
- **Agentic RAG (에이전트 검색 증강 생성)**은 기존의 단순 RAG(Naive RAG) 모델이 가진 한계를 극복하기 위해, 추론(Reasoning) 능력과 행동(Acting) 능력을 갖춘 AI 에이전트를 검색 및 생성 프로세스에 통합한 고급 프레임워크입니다. --> ReAct 

- 쉽게 말해, 단순히 사용자 질문과 관련된 문서를 검색하여 LLM에 넣어주는 대신, LLM 자신이 '이 질문에 답하려면 어떤 정보를, 어디에서, 어떤 순서로 찾아야 할까?'를 스스로 판단하고 실행하게 만듭니다.
#### 도구(Tools)

Agent 가 활용할 도구를 정의하여 Agent 가 추론(reasoning)을 수행할 때 활용하도록 만들 수 있습니다.

Navie RAG아 Advanced RAG 기법으로 구현하기 보다는 최근에는 Agentic RAG 파이프라인을 만들고 있다!!

> 기존 RAG는 단방향이지만, Aentic RAG는 양방향이 가능하다. 

In [ ]:
# web search tool을 만들어보자 
import feedparser # RSS 피드를 읽어서 파이썬 객체로 바꿔주는 라이브러리
from urllib.parse import quote
from typing import List, Dict, Optional
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL


class GoogleNews:
    """
    구글 뉴스를 검색하고 결과를 반환하는 클래스입니다.
    """

    def __init__(self):
        """
        GoogleNews 클래스를 초기화합니다.
        base_url 속성을 설정합니다.
        """
        self.base_url = "https://news.google.com/rss"

    def _fetch_news(self, url: str, k: int = 3) -> List[Dict[str, str]]:
        """
        주어진 URL에서 뉴스를 가져옵니다.

        Args:
            url (str): 뉴스를 가져올 URL
            k (int): 가져올 뉴스의 최대 개수 (기본값: 3)

        Returns:
            List[Dict[str, str]]: 뉴스 제목과 링크를 포함한 딕셔너리 리스트
        """
        news_data = feedparser.parse(url) # RSS url을 읽고 기사 목록 파싱한다. 
        return [
            {"title": entry.title, "link": entry.link}
            for entry in news_data.entries[:k] # entry는 rss의 기사 하나임
        ]

    def _collect_news(self, news_list: List[Dict[str, str]]) -> List[Dict[str, str]]:
        """
        뉴스 리스트를 정리하여 반환합니다.

        Args:
            news_list (List[Dict[str, str]]): 뉴스 정보를 포함한 딕셔너리 리스트

        Returns:
            List[Dict[str, str]]: URL과 내용을 포함한 딕셔너리 리스트
        """
        if not news_list:
            print("해당 키워드의 뉴스가 없습니다.")
            return []

        result = []
        for news in news_list:
            result.append({"url": news["link"], "content": news["title"]})

        return result

    def search_latest(self, k: int = 3) -> List[Dict[str, str]]:
        """
        최신 뉴스를 검색합니다.

        Args:
            k (int): 검색할 뉴스의 최대 개수 (기본값: 3)

        Returns:
            List[Dict[str, str]]: URL과 내용을 포함한 딕셔너리 리스트
        """
        url = f"{self.base_url}?hl=ko&gl=KR&ceid=KR:ko"
        news_list = self._fetch_news(url, k)
        return self._collect_news(news_list)
        # 물론, 이후에 이 함수를 부르지 않아서, 구현만 하고 쓰지는 않는다. 

    def search_by_keyword(
        self, keyword: Optional[str] = None, k: int = 3
    ) -> List[Dict[str, str]]:
        """
        키워드로 뉴스를 검색합니다.

        Args:
            keyword (Optional[str]): 검색할 키워드 (기본값: None)
            k (int): 검색할 뉴스의 최대 개수 (기본값: 3)

        Returns:
            List[Dict[str, str]]: RL과 내용을 포함한 딕셔너U리 리스트
        """
        if keyword:
            encoded_keyword = quote(keyword) # 검색어를 url에 넣을 수 있도록 인코딩
            url = f"{self.base_url}/search?q={encoded_keyword}&hl=ko&gl=KR&ceid=KR:ko"
        else:
            url = f"{self.base_url}?hl=ko&gl=KR&ceid=KR:ko"
        news_list = self._fetch_news(url, k)
        return self._collect_news(news_list)
    
    

# 도구 생성
@tool
def search_news(query: str) -> List[Dict[str, str]]:
    """Search Google News by input keyword"""
    news_tool = GoogleNews()
    return news_tool.search_by_keyword(query, k=5)

# k=5이기 때문에 최종적으로 뉴스 5개 가져오겠지. 근데, k=5를 주지 않고 news_tool.search_by_keyword(query) 하면 위에서 정의한 대로 뉴스 3개를 가져올 것이다. 

In [ ]:
# Retriever를 만들자. 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 로드. 파일의 경로 입력
loader = PyPDFLoader("../data/SPRi AI Brief_6월호_산업동향_F.pdf")

# 텍스트 분할기를 사용하여 문서를 분할합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 문서를 로드하고 분할합니다.
split_docs = loader.load_and_split(text_splitter)

# VectorStore를 생성합니다.
vector = FAISS.from_documents(split_docs, GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

# Retriever를 생성합니다.
retriever = vector.as_retriever()

# RecursiveCharacterTextSplitter에는 기본 separtor (["\n\n", "\n", " ", ""])가 기본적으로 들어가있다. 즉, 따로 지정하지 않아도 내부적으로 문단 기준으로 먼저 자르고, 안되면 줄바꿈 기준, 안되면 공백 기준, 안되면 문자 단위... 이렇게 청킹한다. 

In [ ]:
from langchain_classic.tools.retriever import create_retriever_tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import create_tool_calling_agent
from langchain_classic.agents import AgentExecutor
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory



retriever_tool = create_retriever_tool(  # 우리가 tool을 만들면, tool의 이름과 자세한 정보를 자세하게 기입해야 한다. 이걸 LLM이 보고 tool 선택!
    retriever,
    name="pdf_search",  # 도구의 이름을 입력합니다.
    description="use this tool to search information from the PDF document",  # 도구에 대한 설명을 자세히 기입해야 합니다!!
)

# 우리가 만든 tool!
tools = [search_news, retriever_tool]

# LLM 정의
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# Prompt 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. "
            "Make sure to use the `pdf_search` tool for searching information from the PDF document. "
            "If you can't find the information from the PDF document, use the `search` tool for searching information from the web.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
) # 이렇게 각 tool의 기능을 자세하게 적어놓는 것이 매우 중요하다. 상세하게 적으면 적을수록 llm  Agent가 답변에 대한 tool을 선택할 때 어떤 tool을 선택해야 하는지에 도움이 더 될 것이다. 


# tool calling agent 생성
agent = create_tool_calling_agent(llm, tools, prompt)

# AgentExecutor 생성
# verbose=False 중간 단계 출력 생략
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)


# session_id 를 저장할 딕셔너리 생성
store = {} 
# session id란 대화방 ID! 여러 사용자가 있거나, 한 사용자가 여러 대화를 한다고 할 때 각각의 대화 기록을 구분해야 하기 때문


# session_id 를 기반으로 세션 기록을 가져오는 함수 # 챗봇에서 대화를 기억해야 한다면!
def get_session_history(session_ids):
    if session_ids not in store:  # session_id 가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환


# 채팅 메시지 기록이 추가된 에이전트를 생성합니다.
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # 대화 session_id
    get_session_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
) # 완성! 

# 이렇게 하면 채팅 메세지 기록이 추가된 에이전트를 만들 수 있다. 
# 에이전트가 이렇게 스스로 판단하고 tool을 선택해서 답변하는 과정이 agentic RAG이다. 이게 요새 RAG 트렌드이다. 

In [ ]:
response = agent_with_chat_history.invoke(
    {"input": "2025년 APEC 관련 최신뉴스."},
    # 세션 ID를 설정합니다.
    # 여기서는 간단한 메모리 내 ChatMessageHistory를 사용하기 때문에 실제로 사용되지 않습니다
    config={"configurable": {"session_id": "abc123"}},
)

print(response['output'])

In [ ]:
response = agent_with_chat_history.invoke(
    {"input": "6월호 산업동향에서, 문서에서 찾아서 미국 상무부의 AI 관련 정책에 대해 한글로 정리해줘"},
    # 세션 ID를 설정합니다.
    # 여기서는 간단한 메모리 내 ChatMessageHistory를 사용하기 때문에 실제로 사용되지 않습니다
    config={"configurable": {"session_id": "abc123"}},
)

print(response['output'])

# Agentic RAG 구현 단계 정리

이 노트북의 Agentic RAG는 PDF 문서 검색용 Retriever와 Google News 검색 도구를 하나의 에이전트에 연결하고, LLM이 질문의 성격에 따라 적절한 도구를 선택해 답변하도록 구성한 구조이다.

## 1. 웹 검색 도구 만들기

- `GoogleNews` 클래스를 정의해 Google News RSS에서 최신 뉴스 또는 키워드 기반 뉴스를 가져오도록 만든다.
- `feedparser.parse()`로 RSS 데이터를 읽고, 각 뉴스의 `title`과 `link`를 추출한다.
- `search_by_keyword()`에서는 사용자가 입력한 검색어를 `quote()`로 URL 인코딩한 뒤 Google News 검색 RSS URL에 넣는다.
- 마지막으로 `@tool` 데코레이터를 사용해 `search_news(query)` 함수를 LangChain Tool로 등록한다.
- 이 도구는 문서 내부에 답이 없거나 최신 정보가 필요한 질문이 들어왔을 때 에이전트가 사용할 수 있는 외부 검색 도구 역할을 한다.

## 2. PDF 문서 기반 Retriever 만들기

- `PyPDFLoader`로 `SPRi AI Brief_6월호_산업동향_F.pdf` 파일을 로드한다.
- `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)`로 PDF 내용을 일정 크기의 청크로 나눈다.
- `chunk_overlap=100`을 주어 청크 경계에서 중요한 문맥이 끊기지 않도록 한다.
- `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")`로 각 청크를 임베딩한다.
- `FAISS.from_documents()`로 벡터스토어를 만들고, `vector.as_retriever()`로 검색 가능한 Retriever를 생성한다.
- 이 Retriever는 PDF 안에서 관련 내용을 찾아오는 내부 문서 검색기 역할을 한다.

## 3. Retriever를 Tool로 변환하기

- `create_retriever_tool()`을 사용해 PDF Retriever를 `pdf_search`라는 이름의 Tool로 감싼다.
- Tool 설명에는 `use this tool to search information from the PDF document`라고 적어, LLM이 이 도구를 언제 써야 하는지 판단할 수 있게 한다.
- `tools = [search_news, retriever_tool]`로 웹 검색 도구와 PDF 검색 도구를 하나의 도구 목록에 넣는다.
- 이 단계에서 에이전트는 내부 문서 검색과 외부 뉴스 검색이라는 두 가지 행동 선택지를 갖게 된다.

## 4. LLM과 프롬프트 구성하기

- `ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)`로 답변을 생성하고 도구 호출을 판단할 LLM을 정의한다.
- `temperature=0`은 답변의 일관성과 재현성을 높이기 위한 설정이다.
- `ChatPromptTemplate`의 system 메시지에서 PDF 문서 검색에는 `pdf_search`를 사용하고, PDF에서 찾을 수 없으면 웹 검색 도구를 사용하라고 지시한다.
- 프롬프트에는 `chat_history`, `input`, `agent_scratchpad`가 포함된다.
- `agent_scratchpad`는 에이전트가 중간 추론과 도구 호출 결과를 관리하는 공간이다.

## 5. Tool Calling Agent와 AgentExecutor 생성하기

- `create_tool_calling_agent(llm, tools, prompt)`로 Tool Calling Agent를 만든다.
- 이 에이전트는 사용자 질문을 보고 `pdf_search`를 쓸지, `search_news`를 쓸지, 또는 바로 답변할지 판단한다.
- `AgentExecutor(agent=agent, tools=tools, verbose=False)`로 실제 실행 가능한 에이전트 실행기를 만든다.
- `AgentExecutor`는 LLM 호출, 도구 호출, 도구 결과 반영, 최종 답변 생성을 전체적으로 관리한다.

## 6. 대화 기록을 붙여 에이전트 완성하기

- `store = {}` 딕셔너리에 `session_id`별 대화 기록을 저장한다.
- `get_session_history(session_ids)` 함수는 세션 ID가 처음 들어오면 `ChatMessageHistory()`를 새로 만들고, 이미 있으면 기존 기록을 반환한다.
- `RunnableWithMessageHistory`로 `agent_executor`를 감싸서 대화 기록을 포함한 에이전트를 만든다.
- `input_messages_key="input"`은 사용자 질문이 들어오는 키를 의미한다.
- `history_messages_key="chat_history"`는 프롬프트에 전달할 이전 대화 기록의 키를 의미한다.
- 이 단계까지 완료하면 질문을 이어서 주고받을 수 있는 Agentic RAG 구조가 완성된다.

## 7. 질문 실행하기

- `agent_with_chat_history.invoke()`로 완성된 에이전트에 질문을 입력한다.
- 첫 번째 예시는 `2025년 APEC 관련 최신뉴스.`처럼 최신성이 중요한 질문이므로 웹 검색 도구를 사용할 가능성이 높다.
- 두 번째 예시는 `6월호 산업동향` 문서에서 미국 상무부의 AI 관련 정책을 찾아 정리해 달라는 질문이므로 PDF 검색 도구인 `pdf_search`를 사용할 가능성이 높다.
- `config={"configurable": {"session_id": "abc123"}}`를 전달해 같은 세션의 대화 기록을 유지한다.

## 핵심 동작 흐름

1. 사용자가 질문을 입력한다.
2. LLM이 질문의 의도를 판단한다.
3. PDF 문서 안에서 찾아야 하면 `pdf_search`를 호출한다.
4. 최신 뉴스나 외부 정보가 필요하면 `search_news`를 호출한다.
5. 도구 호출 결과를 바탕으로 LLM이 최종 답변을 생성한다.
6. 세션 ID 기준으로 대화 기록을 저장해 다음 질문에 활용한다.

## 요약

이 코드는 단순히 벡터 DB에서 문서를 검색한 뒤 답변하는 Naive RAG가 아니라, LLM이 상황에 따라 필요한 도구를 직접 선택하는 Agentic RAG 구현이다. PDF에 있는 내용은 `pdf_search` Retriever Tool로 찾고, 최신 정보가 필요한 경우에는 `search_news` 웹 검색 Tool을 사용한다. 즉, 문서 기반 검색과 외부 검색을 모두 도구로 제공하고, 에이전트가 질문에 맞는 검색 경로를 판단해 답변을 생성한다는 점이 핵심이다. 여기에 `RunnableWithMessageHistory`를 붙여 세션별 대화 기록까지 관리하므로, 실제 챗봇 형태로 확장 가능한 Agentic RAG 파이프라인이 된다.
